In [0]:
 
import requests
import time
import uuid
import logging

from pyspark.sql.functions import current_timestamp, current_date, lit
from pyspark.sql.types import StructType


BASE_URL = "https://test-api-2-r15g.onrender.com"

crm_routes = ["customers", "products", "sales"]
erp_routes = ["erp_customers", "erp_locations", "erp_categories"]

BRONZE_DB = "sqldatawarehouse.bronze"
AUDIT_TABLE = f"{BRONZE_DB}.ingestion_audit"

batch_id = str(uuid.uuid4())


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion")


In [0]:


def fetch_api_data(url, retries=3):
    for i in range(retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            if isinstance(data, list):
                return data
            else:
                logger.warning(f"Unexpected response format: {url}")
                return None

        except Exception as e:
            logger.warning(f"Attempt {i+1} failed for {url}: {e}")
            time.sleep(2 ** i)

    return None



In [0]:

from datetime import datetime

def log_audit(route, table_name, row_count, status):
    audit_df = spark.createDataFrame([(
        route,
        table_name,
        row_count,
        batch_id,
        status,
        datetime.now()
    )], [
        "route",
        "table_name",
        "row_count",
        "batch_id",
        "status",
        "ingestion_time"
    ])

    audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)



In [0]:
from pyspark.sql.types import StructType, StructField, StringType

def normalize_records(records):
    """
    Ensures all dicts have same keys and values are string-safe
    """
    if not records:
        return records

    # collect all possible keys
    all_keys = set()
    for r in records:
        all_keys.update(r.keys())

    normalized = []
    for r in records:
        new_r = {}
        for k in all_keys:
            val = r.get(k, None)

            # force everything to string OR None
            if isinstance(val, (dict, list)):
                new_r[k] = str(val)
            else:
                new_r[k] = val

        normalized.append(new_r)

    return normalized

In [0]:

def ingest_api_data(route, table_prefix):

    url = f"{BASE_URL}/api/{route}"
    table_name = f"{BRONZE_DB}.{table_prefix}_{route}"

    try:
        records = fetch_api_data(url)

        if not records:
            logger.info(f"No data for {route}")
            log_audit(route, table_name, 0, "EMPTY")
            return

        # Create DataFrame
        records = normalize_records(records)
        df = spark.createDataFrame(records)

        # Enrich metadata (Bronze standard)
        df = (
            df.withColumn("ingestion_time", current_timestamp())
              .withColumn("ingestion_date", current_date())
              .withColumn("batch_id", lit(batch_id))
              .withColumn("source_system", lit(table_prefix))
              .withColumn("source_route", lit(route))
        )

        # Write to Delta (Bronze = append-only)
        (
            df.write
            .format("delta")
            .mode("append")
            .partitionBy("ingestion_date")
            .saveAsTable(table_name)
        )

        logger.info(f"SUCCESS → {table_name} | rows={df.count()}")

        log_audit(route, table_name, df.count(), "SUCCESS")

    except Exception as e:
        logger.error(f"FAILED → {route}: {e}")
        log_audit(route, table_name, 0, f"FAILED: {str(e)[:200]}")



In [0]:

for route in crm_routes:
    ingest_api_data(route, "crm")

for route in erp_routes:
    ingest_api_data(route, "erp")

In [0]:
# %sql
# drop table sqldatawarehouse.bronze.api_customers;
# drop table sqldatawarehouse.bronze.crm_customers;
# drop table sqldatawarehouse.bronze.crm_products;
# drop table sqldatawarehouse.bronze.crm_sales;
# drop table sqldatawarehouse.bronze.erp_erp_categories;
# drop table sqldatawarehouse.bronze.erp_erp_customers;
# drop table sqldatawarehouse.bronze.erp_erp_locations;